# UWB-MoE: Modular Mixture of Experts for Indoor Localization

This demonstration notebook walks through the core workflow of **UWB-MoE**:
1. **Configuration**: Loading declarative YAML settings.
2. **Data Pipeline**: Preprocessing raw Channel Impulse Response (CIR) and Time Difference (TD) telemetry.
3. **Model Construction**: Instantiating the deep Mixture-of-Experts architecture.
4. **Training**: Optimizing localization loss with load-balancing diversity regularization.
5. **Evaluation & Explainability**: Analyzing localization RMSE, percentiles, and gate activation histograms.


In [ ]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import torch
import numpy as np
import matplotlib.pyplot as plt

from uwb_moe.config import load_config
from uwb_moe.data.processor import UWBDataProcessor
from uwb_moe.data.dataset import create_dataloaders
from uwb_moe.models.moe_model import UWBMoEModel, count_parameters
from uwb_moe.pipelines.train import Trainer
from uwb_moe.pipelines.evaluate import Evaluator
from uwb_moe.utils.seed import seed_everything

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")


## 1. Configuration & Reproducibility
We set a global random seed and load experiment settings from YAML.

In [ ]:
config = load_config(project_root / 'configs' / 'synthetic.yaml')
seed_everything(config.train.seed)
print("Loaded configuration:", config.model)


## 2. Data Preparation & Normalization
Generate synthetic multi-anchor measurements (or use `data/raw/uwb_test_data.hdf5` if available).

In [ ]:
from tests.conftest import create_synthetic_hdf5

data_path = project_root / 'data' / 'demo_synthetic.hdf5'
create_synthetic_hdf5(data_path, num_samples=60, num_anchors=4, cir_len=config.data.cir_length)

processor = UWBDataProcessor(cir_length=config.data.cir_length)
features, targets = processor.process_dataset(data_path, is_training=True)

print(f"Samples: {len(targets)}")
print(f"Anchors detected: {processor.n_anchors}")
print(f"CIR tensor shape: {features['cir'].shape}")
print(f"TD tensor shape:  {features['td'].shape}")
print(f"Target shape:     {targets.shape}")


## 3. DataLoader Construction

In [ ]:
train_loader, val_loader = create_dataloaders(
    features=features,
    targets=targets,
    batch_size=8,
    val_split_ratio=0.25,
    num_workers=0,
)

sample_batch = next(iter(train_loader))
print("Batch CIR shape:   ", sample_batch['cir'].shape)
print("Batch Target shape:", sample_batch['target'].shape)


## 4. Model Architecture & Parameter Breakdown

In [ ]:
model = UWBMoEModel(
    cir_length=config.model.cir_length,
    n_anchors=processor.n_anchors,
    embedding_dim=config.model.embedding_dim,
    feature_dim=config.model.feature_dim,
    gate_hidden_dim=config.model.gate_hidden_dim,
    num_experts=config.model.num_experts,
    expert_hidden_dim=config.model.expert_hidden_dim,
    position_head_hidden=config.model.position_head_hidden,
    output_dim=config.model.output_dim,
    diversity_loss_coeff=config.model.diversity_loss_coeff,
)

param_counts = count_parameters(model)
print(f"{'Component':<25} | {'Parameters':<10}")
print("-" * 40)
for comp, cnt in param_counts.items():
    print(f"{comp:<25} | {cnt:<10,}")


## 5. End-to-End Training

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    config=config,
    device=device,
)

history = trainer.run()


## 6. Evaluation & Diagnostics
Evaluate the model across validation bursts, compute metrics, and analyze expert gating.

In [ ]:
evaluator = Evaluator(model=model, data_loader=val_loader, device=device, output_dir='outputs/notebook_eval')
metrics, y_true, y_pred, gate_values = evaluator.run()

print(metrics.summary())


### 6.1 NLOS Gate Distribution
Histogram showing the distribution of routing probabilities assigned to the NLOS expert.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(gate_values, bins=30, color='#3498db', edgecolor='#2c3e50', alpha=0.85)
plt.title('NLOS Gate Value Distribution', fontsize=13, fontweight='bold')
plt.xlabel('NLOS Expert Weight', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


### 6.2 Ground Truth vs. Predicted Positions

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_true[:, 0], y_true[:, 1], c='#2ecc71', label='Ground Truth', s=40)
plt.scatter(y_pred[:, 0], y_pred[:, 1], c='#e74c3c', label='Predicted', marker='x', s=40)
for i in range(len(y_true)):
    plt.plot([y_true[i, 0], y_pred[i, 0]], [y_true[i, 1], y_pred[i, 1]], 'gray', alpha=0.3)
plt.title('2D Positioning Comparison', fontsize=13, fontweight='bold')
plt.xlabel('X (meters)', fontsize=11)
plt.ylabel('Y (meters)', fontsize=11)
plt.legend()
plt.axis('equal')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
